# glasses3d on Colab

Reconstruct a 3D world from Meta Ray-Ban glasses footage using a Colab GPU.

**Set the runtime first:** Runtime -> Change runtime type -> GPU.

Two modes:

| Mode | What it needs | Works on |
|---|---|---|
| **Offline** (recommended) | a recorded clip | any GPU, including a free T4 |
| **Live** | phone streaming through a tunnel | L4 / A100 or better |

Offline is not the fallback — it is the better path. Recorded clips are 3K/60,
about **9x the pixels** of the 720p live stream, and there is no latency budget
to fight. Live is for interactivity, not fidelity.


## 1. What GPU did we get?

Colab allocates opportunistically — the same notebook gets a T4 one run
and an L4 the next — so capability has to be measured, not assumed.


In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or
      'No GPU. Runtime > Change runtime type > GPU, then rerun.')


## 2. Install

First run takes ~5-10 minutes. Mounting Drive caches the repo and the
model weights so later sessions skip most of it — worth doing, because a
free-tier session is capped at 12 hours and dies after 90 minutes idle.


In [ ]:
#@title Mount Drive (optional but recommended)
USE_DRIVE = True  #@param {type:'boolean'}

import os
ROOT = '/content'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/glasses3d-workspace'
    os.makedirs(ROOT, exist_ok=True)
print('workspace:', ROOT)


In [ ]:
#@title Get the code
REPO = os.path.join(ROOT, 'glasses3d')

# Bring your own copy. Push ~/projects/glasses3d somewhere you can clone, or
# upload the server/ and tools/ directories straight into REPO.
GIT_URL = ''  #@param {type:'string'}

if GIT_URL:
    if os.path.isdir(os.path.join(REPO, '.git')):
        !cd {REPO} && git pull --rebase
    else:
        !git clone {GIT_URL} {REPO}
elif not os.path.isdir(REPO):
    os.makedirs(os.path.join(REPO, 'server'), exist_ok=True)
    print('Created', REPO)
    print('Upload the server/ directory into it (Files pane), or set GIT_URL above.')

assert os.path.isdir(REPO), REPO
sys.path.insert(0, os.path.join(REPO, 'server'))
print('repo:', REPO)


In [ ]:
#@title Install dependencies (~5-10 min first run)
!pip -q install opencv-python-headless websockets

# MapAnything. The Apache-2.0 weights are the default in backends.py; the
# other variant is CC-BY-NC and research-only.
MA = os.path.join(ROOT, 'map-anything')
if not os.path.isdir(MA):
    !git clone -q https://github.com/facebookresearch/map-anything {MA}
!pip -q install -e {MA}
print('installed')


## 3. Capability check

Tells you which mode is actually viable on the GPU you were assigned,
instead of letting you discover it three steps later.


In [ ]:
from backends import describe_gpu

gpu = describe_gpu()
for k, v in gpu.items():
    print(f'{k:22} {v}')

if not gpu['available']:
    print('\nNo CUDA. Runtime > Change runtime type > GPU.')
elif gpu['live_viable']:
    print(f"\n{gpu['name']}: live mode should work. Offline still gives better quality.")
else:
    print(f"\n{gpu['name']}: OFFLINE ONLY.")
    print(f"Estimated {gpu['expected_slam_fps']} fps for live tracking; ~8 is the floor")
    print('for it to feel live. Use offline mode — better input, no latency budget.')


---
# Offline mode

Upload a clip, get `scene.ply` back.


### 3a. Calibration (do this once)

The glasses use a 12MP ultrawide with strong barrel distortion. Skipping
this produces warped geometry and drifting scale — it is the highest
quality-per-effort step in the whole pipeline.

Record ~30s of a 9x6-inner-corner checkerboard, working it into the frame
**corners** where distortion is strongest. Then upload it here.

Already calibrated? Upload `intrinsics.json` to `REPO/calib/` and skip.


In [ ]:
#@title Calibrate from a checkerboard video
from google.colab import files

os.makedirs(os.path.join(REPO, 'calib'), exist_ok=True)
print('Upload the checkerboard video (or press Cancel to skip):')
up = files.upload()

if up:
    board = os.path.join(REPO, 'calib', list(up)[0])
    open(board, 'wb').write(list(up.values())[0])
    !cd {REPO} && python3 server/calibrate.py --video {board} --square-mm 25
else:
    print('Skipped. Reconstruction will run on distorted frames.')


### 3b. Reconstruct


In [ ]:
#@title Upload a clip
from google.colab import files

print('Upload your glasses clip:')
up = files.upload()
assert up, 'no file uploaded'

clip = os.path.join(ROOT, list(up)[0])
open(clip, 'wb').write(list(up.values())[0])
print('%s (%.1f MB)' % (clip, os.path.getsize(clip) / 1e6))


In [ ]:
#@title Run reconstruction
VIEWS = 32       #@param {type:'integer'}
WIDTH = 518      #@param {type:'integer'}
MIN_CONF = 0.5   #@param {type:'number'}
MAX_POINTS = 1500000  #@param {type:'integer'}

OUT = os.path.join(ROOT, 'out')

# VIEWS drives both quality and VRAM. 32 is comfortable on a T4; push it up on
# an A100. If this OOMs, halve VIEWS before touching anything else.
cmd = (f'cd {REPO} && python3 server/reconstruct.py'
       f' --video "{clip}" --out "{OUT}" --backend mapanything'
       f' --views {VIEWS} --width {WIDTH}'
       f' --min-conf {MIN_CONF} --max-points {MAX_POINTS}')
print(cmd)
!{cmd}


In [ ]:
#@title Inspect and download scene.ply
import json
meta = json.load(open(os.path.join(OUT, 'meta.json')))
for k, v in meta.items():
    print(f'{k:16} {v}')

from google.colab import files
files.download(os.path.join(OUT, 'scene.ply'))


### 3c. Where scene.ply goes

| Target | How |
|---|---|
| Web | drag onto [superspl.at/editor](https://superspl.at/editor) — do this first, fastest feedback |
| Blender | 3DGS Render addon (KIRI) -> Import PLY |
| Unity | [UnityGaussianSplatting](https://github.com/aras-p/UnityGaussianSplatting) |
| Unreal | [SplatRenderer](https://github.com/DazaiStudio/SplatRenderer-UEPlugin), UE 5.5+ |
| Quest VR | via Unity — keep under ~400k splats for 72fps |


---
# Live mode

Only worth attempting if the capability check said `live_viable`.

**The problem:** Colab has no public inbound address, so the phone cannot
reach it directly. A tunnel is mandatory. That adds 100-300 ms on top of
the glasses -> phone hop, so budget for latency well above the 200 ms
target that a local GPU box would hit.


In [ ]:
#@title Open a public tunnel
import re, time, subprocess

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

PORT = 8765
log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=log, stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        url = m.group(0)
        break

if url:
    print('Tunnel up.')
    print('Set serverURL in Glasses3DRelay.swift to:')
    print('   ' + url.replace('https://', 'wss://'))
else:
    print('Tunnel did not come up — check /content/tunnel.log')


In [ ]:
#@title Run the ingest server
# Runs until you interrupt it. The phone connects to the wss:// URL above.
!cd {REPO} && python3 -u server/ingest.py --source ws --port 8765


### Live-mode reality check

- **Free-tier T4 will not keep up.** Roughly 2-3 fps for dense SLAM against
  the ~8 fps floor for it to feel live.
- **Sessions die.** 90 minutes idle, 12 hours absolute on free tier. The
  tunnel URL changes every restart, so the phone needs reconfiguring.
- **Glasses battery** is the other clock. Design for 2-5 minute captures.

If live is the actual goal rather than a demo, a persistent GPU box
(RunPod, Lambda) with a stable address is a much better fit than Colab.
Colab is excellent at the offline path and awkward at this one.
